# MTAOS Per-Sensor Zernike Coefficients — EFD Time Series

Queries `lsst.sal.MTAOS.logevent_wavefrontError` from the EFD to retrieve the
measured annular Zernike coefficients reported by the AOS wavefront estimation
pipeline (WEP) for each of the four corner wavefront sensors.

Each event is emitted once per corner sensor after a pair of intra/extra-focal
donut images are processed.  Fields:

| EFD field | Noll index | Aberration |
|---|---|---|
| `annularZernikeCoeff0` | Z4 | Defocus |
| `annularZernikeCoeff1` | Z5 | Oblique astigmatism |
| `annularZernikeCoeff2` | Z6 | Vertical astigmatism |
| `annularZernikeCoeff3` | Z7 | Vertical coma |
| `annularZernikeCoeff4` | Z8 | Horizontal coma |
| … | … | … |
| `annularZernikeCoeff18` | Z22 | — |

The `sensorId` field identifies which corner sensor produced the measurement.
Typical LSSTCam corner sensor IDs are mapped in the **Configuration** cell below.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from astropy.time import Time, TimeDelta
import astropy.units as u

from lsst_efd_client import EfdClient

%matplotlib inline

## Configuration

In [ ]:
# ── Night to query ────────────────────────────────────────────────────────────
# Set to the TAI date of the night (evening date, YYYY-MM-DD).
# The window covers 12:00 UTC on that date to 12:00 UTC the next day,
# which brackets a full Rubin observing night.
NIGHT_DATE = "2026-06-07"  # <-- change this

t_start = Time(f"{NIGHT_DATE}T12:00:00", scale="utc")
t_end = t_start + TimeDelta(1 * u.day)
print(f"Query window: {t_start.iso}  →  {t_end.iso}  (UTC)")

# ── EFD ───────────────────────────────────────────────────────────────────────
EFD_ALIAS = "usdf_efd"
WFE_TOPIC = "lsst.sal.MTAOS.logevent_wavefrontError"

# EFD stores Zernike coefficients as:
#   nollZernikeValues0  → Z4  (Noll index 4, defocus)
#   nollZernikeValues1  → Z5  (oblique astigmatism)
#   nollZernikeValues2  → Z6  (vertical astigmatism)
#   ...
#   nollZernikeValues24 → Z28
N_NOLL_VALUES = 25  # number of Zernike values stored (Z4–Z28)
NOLL_OFFSET = 4  # first Noll index

# ── Corner sensor ID → label mapping ─────────────────────────────────────────
SENSOR_LABELS = {
    191: "R00 (SW corner)",
    195: "R04 (SE corner)",
    199: "R40 (NW corner)",
    203: "R44 (NE corner)",
}
SENSOR_COLORS = {
    191: "steelblue",
    195: "tomato",
    199: "mediumseagreen",
    203: "darkorchid",
}

## Connect to EFD

In [ ]:
efd_client = EfdClient(EFD_ALIAS)
print(f"Connected to EFD alias: {EFD_ALIAS}")

## Query `logevent_wavefrontError`

In [ ]:
# Fetch all wavefront error events for the night.
# Each row = one sensor per AOS cycle.
# Zernike data is in nollZernikeValues0-24 (Noll Z4-Z28)
# with corresponding nollZernikeIndices0-24 confirming the Noll index.
noll_val_fields = [f"nollZernikeValues{i}" for i in range(N_NOLL_VALUES)]
noll_idx_fields = [f"nollZernikeIndices{i}" for i in range(N_NOLL_VALUES)]

df_wfe = await efd_client.select_time_series(
    WFE_TOPIC,
    fields=["sensorId", "visitId"] + noll_val_fields + noll_idx_fields,
    start=t_start,
    end=t_end,
)

print(f"Total rows returned: {len(df_wfe)}")
if df_wfe.empty:
    print("No data — check NIGHT_DATE or the EFD alias.")
else:
    print(f"Time range: {df_wfe.index.min()}  →  {df_wfe.index.max()}")
    print(f"Unique sensorIds found: {sorted(df_wfe['sensorId'].unique())}")
    # Confirm Noll index mapping from first row
    print(
        f"\nNoll index mapping (first row): Z{int(df_wfe['nollZernikeIndices0'].iloc[0])} "
        f"to Z{int(df_wfe[f'nollZernikeIndices{N_NOLL_VALUES-1}'].iloc[0])}"
    )
    print(
        f"Z4 (nollZernikeValues0) range: "
        f"{df_wfe['nollZernikeValues0'].min():.3f} to {df_wfe['nollZernikeValues0'].max():.3f} µm"
    )
    print("\nFirst few rows:")
    display(
        df_wfe[
            [
                "sensorId",
                "visitId",
                "nollZernikeValues0",
                "nollZernikeValues1",
                "nollZernikeValues2",
            ]
        ].head(8)
    )

### Sensor ID inspection

If the sensor IDs above differ from the defaults in `SENSOR_LABELS`, update the
mapping in the **Configuration** cell and re-run from there.

In [ ]:
# Per-sensor event count and Z4 (defocus) statistics
if not df_wfe.empty:
    z4_col = "nollZernikeValues0"  # Noll Z4 = defocus
    summary = (
        df_wfe.groupby("sensorId")[z4_col]
        .agg(N="count", mean="mean", std="std", min="min", median="median", max="max")
        .rename_axis("sensorId")
    )
    summary.insert(
        0,
        "label",
        summary.index.map(lambda s: SENSOR_LABELS.get(int(s), f"sensor {s}")),
    )
    print("Z4 (nollZernikeValues0, Noll defocus) statistics per sensor (µm):")
    display(summary.round(4))

## Time Series: Z4 (Defocus) per Corner Sensor

In [ ]:
if df_wfe.empty:
    print("No data to plot.")
else:
    z4_col = "nollZernikeValues0"  # Noll Z4 defocus

    sensor_ids = sorted(df_wfe["sensorId"].unique())
    n_sensors = len(sensor_ids)

    fig, axes = plt.subplots(
        n_sensors,
        1,
        figsize=(13, 3.5 * n_sensors),
        sharex=True,
        squeeze=False,
    )

    for ax, sid in zip(axes[:, 0], sensor_ids):
        df_s = df_wfe[df_wfe["sensorId"] == sid].sort_index()
        label = SENSOR_LABELS.get(int(sid), f"sensor {sid}")
        color = SENSOR_COLORS.get(int(sid), "gray")

        ax.plot(
            df_s.index, df_s[z4_col], marker="o", ms=4, lw=1.2, color=color, label=label
        )
        ax.axhline(0, color="black", lw=0.7, ls="--", alpha=0.5)

        median_val = df_s[z4_col].median()
        ax.axhline(
            median_val,
            color=color,
            lw=1.0,
            ls=":",
            label=f"median = {median_val:.4f} µm",
        )

        ax.set_ylabel("Z4 (µm)")
        ax.set_title(label, fontsize=10, loc="left")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.3)

    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[-1, 0].set_xlabel(f"UTC time on {NIGHT_DATE}")

    fig.suptitle(
        f"MTAOS Per-Sensor Z4 (Defocus) — {NIGHT_DATE}\n"
        f"nollZernikeValues0  [µm, Noll Z4]",
        fontsize=12,
    )
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## All Four Sensors Overlaid

In [ ]:
if not df_wfe.empty:
    fig, ax = plt.subplots(figsize=(13, 4.5))

    for sid in sorted(df_wfe["sensorId"].unique()):
        df_s = df_wfe[df_wfe["sensorId"] == sid].sort_index()
        label = SENSOR_LABELS.get(int(sid), f"sensor {sid}")
        color = SENSOR_COLORS.get(int(sid), "gray")
        ax.plot(
            df_s.index,
            df_s["nollZernikeValues0"],
            marker="o",
            ms=3,
            lw=1.0,
            alpha=0.8,
            color=color,
            label=label,
        )

    ax.axhline(0, color="black", lw=0.7, ls="--", alpha=0.5)
    ax.set_ylabel("Z4 Defocus (µm)")
    ax.set_xlabel(f"UTC time on {NIGHT_DATE}")
    ax.set_title(f"MTAOS Z4 — all corner sensors — {NIGHT_DATE}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## Bonus: Multiple Zernike Modes per Sensor

Grid of subplots — one row per Zernike mode, one column per sensor.

In [ ]:
# Which Noll indices to show (subset to keep the figure readable)
# Each Noll index N maps to nollZernikeValues{N - NOLL_OFFSET}
NOLL_MODES = [4, 5, 6, 7, 8, 11, 17]  # defocus, astig, coma, spherical, 2nd horiz coma

NOLL_NAMES = {
    4: "Z4 defocus",
    5: "Z5 oblique astig",
    6: "Z6 vert astig",
    7: "Z7 vert coma",
    8: "Z8 horiz coma",
    9: "Z9",
    10: "Z10",
    11: "Z11 spherical",
    12: "Z12",
    13: "Z13",
    17: "Z17 2nd horiz coma",
}

if not df_wfe.empty:
    sensor_ids = sorted(df_wfe["sensorId"].unique())
    n_modes = len(NOLL_MODES)
    n_sensors = len(sensor_ids)

    fig, axes = plt.subplots(
        n_modes,
        n_sensors,
        figsize=(4.5 * n_sensors, 3.0 * n_modes),
        sharex="col",
        sharey="row",
        squeeze=False,
    )

    for col_idx, sid in enumerate(sensor_ids):
        df_s = df_wfe[df_wfe["sensorId"] == sid].sort_index()
        label = SENSOR_LABELS.get(int(sid), f"sensor {sid}")
        color = SENSOR_COLORS.get(int(sid), "gray")

        for row_idx, noll in enumerate(NOLL_MODES):
            ax = axes[row_idx][col_idx]
            col = f"nollZernikeValues{noll - NOLL_OFFSET}"

            if col not in df_s.columns:
                ax.set_visible(False)
                continue

            ax.plot(df_s.index, df_s[col], marker="o", ms=2, lw=0.9, color=color)
            ax.axhline(0, color="black", lw=0.5, ls="--", alpha=0.4)
            ax.grid(True, alpha=0.25)

            if row_idx == 0:
                ax.set_title(label, fontsize=9)
            if col_idx == 0:
                ax.set_ylabel(f"{NOLL_NAMES.get(noll, f'Z{noll}')} (µm)", fontsize=8)
            if row_idx == n_modes - 1:
                ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
                ax.set_xlabel("UTC", fontsize=8)
                plt.setp(
                    ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7
                )

    fig.suptitle(
        f"MTAOS Annular Zernike Coefficients per Corner Sensor — {NIGHT_DATE}",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

## AOS Cycle Sequencing — Convergence Diagnostic

Each `logevent_wavefrontError` record is one AOS cycle (one row per corner sensor
at the same timestamp). Group consecutive cycles into **sequences** separated by
gaps larger than `SEQUENCE_GAP_MIN` minutes, then index each cycle within its
sequence (0, 1, 2, ...). This lets us ask: *does the residual shrink with cycle
number (loop converging) or stay flat (drift / overshoot dominates)?*


In [ ]:
# Gap (minutes) above which we start a new sequence.
# Typical AOS cycle period is ~1.5 min (median gap); gaps > 2 min indicate a
# pause (slew, target change). The night's gap distribution clusters tightly at
# 1.5-1.7 min with a tail above 2 min — at threshold 2.0 we get O(20) sequences.
SEQUENCE_GAP_MIN = 2.0

if df_wfe.empty:
    print("No data to sequence.")
else:
    # Per-cycle timestamp = first sensor's timestamp for that visitId.
    cycle_times = df_wfe.reset_index().groupby("visitId")["index"].min().sort_values()
    dt_min = cycle_times.diff().dt.total_seconds().div(60.0)
    seq_id = (dt_min.fillna(SEQUENCE_GAP_MIN + 1) > SEQUENCE_GAP_MIN).cumsum()
    cycle_index = seq_id.groupby(seq_id).cumcount()

    cycle_meta = pd.DataFrame(
        {
            "cycle_time": cycle_times,
            "dt_prev_min": dt_min,
            "sequence_id": seq_id,
            "cycle_index": cycle_index,
        }
    )

    # Attach to the per-sensor frame.
    df_wfe = df_wfe.drop(
        columns=["sequence_id", "cycle_index", "dt_prev_min", "cycle_time"],
        errors="ignore",
    )
    df_wfe = df_wfe.merge(cycle_meta, left_on="visitId", right_index=True, how="left")

    n_seq = cycle_meta["sequence_id"].nunique()
    seq_lengths = cycle_meta.groupby("sequence_id").size()
    print(f"Identified {n_seq} sequences (gap > {SEQUENCE_GAP_MIN} min).")
    print(
        f"Cycles per sequence: min={seq_lengths.min()}, "
        f"median={int(seq_lengths.median())}, max={seq_lengths.max()}"
    )
    print(f"Total cycles: {len(cycle_meta)}")
    display(
        seq_lengths.value_counts()
        .sort_index()
        .rename_axis("cycles_in_seq")
        .to_frame("n_sequences")
        .head(15)
    )

## Zernike Residual vs Cycle Index Within a Sequence

For each sensor, plot one faded line per sequence (Zernike value vs cycle index)
and overlay the mean ± std across sequences. If the loop is converging the band
should narrow toward zero with increasing cycle number; if drift / overshoot
dominates we expect a roughly flat scatter band.

What to look for:

- **Z4** — expect rapid narrowing in the first 2–3 cycles as the loop pulls
  defocus toward zero, then a steady-state scatter band set by per-cycle drift
  / overshoot (telescope settling, mount motion across the 30 s exposures).
  A wide steady-state band with a near-zero mean → loop is finding zero on
  average but each individual cycle is noisy.
- **Z17** — if the offset persists at *all* cycle indices (including high N),
  it is *not* a convergence artefact: either Z17 is not in the corrected modal
  basis, is under-weighted in the modal decomposition, or there is a
  reference / calibration bias. A converging loop would scatter, not bias.


In [ ]:
# Modes to inspect — Z4 (defocus, control loop's main term), Z8 (horiz coma),
# Z17 (2nd horiz coma — the one with the apparent offset).
CONV_MODES = [4, 8, 17]
CONV_NAMES = {
    4: "Z4 defocus",
    8: "Z8 horiz coma",
    17: "Z17 2nd horiz coma",
}
MIN_SEQ_LEN = 2  # need >= 2 cycles to see evolution
MIN_COUNT = 8  # require >= this many sensor-sequences at a given cycle index

if df_wfe.empty:
    print("No data.")
else:
    # Keep sequences with >= MIN_SEQ_LEN cycles.
    seq_lengths = df_wfe.groupby("sequence_id")["cycle_index"].max() + 1
    long_seqs = seq_lengths[seq_lengths >= MIN_SEQ_LEN].index
    df_conv = df_wfe[df_wfe["sequence_id"].isin(long_seqs)].copy()

    sensor_ids = sorted(df_conv["sensorId"].unique())
    n_sensors = len(sensor_ids)
    n_modes = len(CONV_MODES)

    fig, axes = plt.subplots(
        n_modes,
        n_sensors,
        figsize=(4.0 * n_sensors, 2.8 * n_modes),
        sharex=True,
        sharey="row",
        squeeze=False,
    )

    for col_idx, sid in enumerate(sensor_ids):
        df_s = df_conv[df_conv["sensorId"] == sid]
        label = SENSOR_LABELS.get(int(sid), f"sensor {sid}")
        color = SENSOR_COLORS.get(int(sid), "gray")

        for row_idx, noll in enumerate(CONV_MODES):
            ax = axes[row_idx][col_idx]
            col = f"nollZernikeValues{noll - NOLL_OFFSET}"
            if col not in df_s.columns:
                ax.set_visible(False)
                continue

            # Faded per-sequence trajectories.
            for sq, df_sq in df_s.groupby("sequence_id"):
                df_sq = df_sq.sort_values("cycle_index")
                ax.plot(
                    df_sq["cycle_index"],
                    df_sq[col],
                    color=color,
                    alpha=0.18,
                    lw=0.8,
                    marker="o",
                    ms=2,
                )

            # Mean +/- std across sequences at each cycle index.
            agg = df_s.groupby("cycle_index")[col].agg(["mean", "std", "count"])
            agg = agg[agg["count"] >= 2]
            if not agg.empty:
                ax.plot(
                    agg.index,
                    agg["mean"],
                    color=color,
                    lw=2.0,
                    marker="o",
                    ms=4,
                    label="mean",
                )
                ax.fill_between(
                    agg.index,
                    agg["mean"] - agg["std"],
                    agg["mean"] + agg["std"],
                    color=color,
                    alpha=0.18,
                    label=r"$\pm$1$\sigma$",
                )

            ax.axhline(0, color="black", lw=0.5, ls="--", alpha=0.5)
            ax.grid(True, alpha=0.25)

            if row_idx == 0:
                ax.set_title(label, fontsize=9)
            if col_idx == 0:
                ax.set_ylabel(f"{CONV_NAMES.get(noll, f'Z{noll}')} (µm)", fontsize=9)
            if row_idx == n_modes - 1:
                ax.set_xlabel("cycle index within sequence", fontsize=9)
            if row_idx == 0 and col_idx == n_sensors - 1:
                ax.legend(fontsize=7, loc="upper right")

    fig.suptitle(
        rf"AOS Zernike Residual vs Cycle-in-Sequence — {NIGHT_DATE}  "
        rf"(sequences with $\geq${MIN_SEQ_LEN} cycles, gap > {SEQUENCE_GAP_MIN} min)",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

    # Per-mode summary: residual mean / std vs cycle index, pooled across sensors.
    print(
        "\nPer-mode residual statistics by cycle_index "
        f"(only cycle indices with count >= {MIN_COUNT}):"
    )
    for noll in CONV_MODES:
        col = f"nollZernikeValues{noll - NOLL_OFFSET}"
        s = df_conv.groupby("cycle_index")[col].agg(["mean", "std", "count"]).round(4)
        s = s[s["count"] >= MIN_COUNT]
        print(f"\n--- {CONV_NAMES.get(noll, f'Z{noll}')} ({col}) ---")
        if len(s) <= 14:
            print(s.to_string())
        else:
            print(s.head(7).to_string())
            print("  ...")
            print(s.tail(5).to_string())